In [1]:
import os
import statistics
from collections import Counter
from datetime import datetime, timezone
from json import dumps
import pandas as pd
import numpy as np
from pymongo import MongoClient
from dotenv import dotenv_values


In [2]:
# 1. Chargement ciblé des variables d'environnement
env_vars = dotenv_values(".env")
env_local_vars = dotenv_values(".env.local")

atlas_uri = env_vars.get("ATLAS_URI")
local_uri = env_local_vars.get("LOCAL_URI")

# 3. Test de la connexion cloud (Atlas)
try:
    if not atlas_uri:
        raise ValueError("ATLAS_URI non trouvé dans .env")
    client = MongoClient(atlas_uri, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    db = client["securite_routiere"]
    print("Connexion cloud établie avec succès")
except Exception as e:
    print(f"Erreur de connexion cloud : {e}")

Connexion cloud établie avec succès


In [ ]:
# Récupération des 5 premiers éléments de la collection "accidents" pour vérification
try:
    accidents_collection = db["accidents"]
    sample_accidents = accidents_collection.find().limit(5)
    print("Exemple d'accidents récupérés :")
    for accident in sample_accidents:
        print(accident)
except Exception as e:
    print(f"Erreur lors de la récupération des accidents : {e}")

In [ ]:
# Test de recherche avec un index sur le champ "Num_Acc"
# TODO db.accidents.find({}, {"Num_Acc": 1}).explain("executionStats").executionStats.executionStages

In [4]:
# Création des index pour améliorer les performances des requêtes
try:
    # Création des index
    accidents_collection.create_index("Num_Acc", unique=True)
    accidents_collection.create_index("dep")
    accidents_collection.create_index([("localisation", "2dsphere")])
    print("Index créés avec succès")
except Exception as e:
    print(f"Erreur lors de la création des index : {e}")

Erreur lors de la création des index : Index build failed: 9e5f398b-d90b-40f9-b6c5-6a962f82a6e6: Collection securite_routiere.accidents ( 42e8fc5c-3cc1-4469-af6d-2f330c5f7ffb ) :: caused by :: E11000 duplicate key error collection: securite_routiere.accidents index: Num_Acc_1 dup key: { Num_Acc: 202400000001 }, full error: {'ok': 0.0, 'errmsg': 'Index build failed: 9e5f398b-d90b-40f9-b6c5-6a962f82a6e6: Collection securite_routiere.accidents ( 42e8fc5c-3cc1-4469-af6d-2f330c5f7ffb ) :: caused by :: E11000 duplicate key error collection: securite_routiere.accidents index: Num_Acc_1 dup key: { Num_Acc: 202400000001 }', 'code': 11000, 'codeName': 'DuplicateKey', 'keyPattern': {'Num_Acc': 1}, 'keyValue': {'Num_Acc': 202400000001}, '$clusterTime': {'clusterTime': Timestamp(1787908379, 6), 'signature': {'hash': b':\x90\xc3\xd8]\xb8\xbf\xdaV\xe1_\x9c\x90\x9fg\x12\xcb\x99\x88(', 'keyId': 7643078544444096515}}, 'operationTime': Timestamp(1787908379, 6)}


In [ ]:
# Test de recherche avec un index sur le champ "Num_Acc"
# TODO db.accidents.find({}, {"Num_Acc": 1}).explain("executionStats").executionStats.executionStages